In [42]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [43]:
#Extract text from pdf
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents=loader.load()
    return documents

In [44]:
import os
print(os.getcwd())

d:\Soham_Jadhav\Medical_Chatbot_LLM\research


In [45]:
extracted_data=load_pdf_files("../data")
print(extracted_data[0].page_content[:5000])

Soham Rajesh Jadhav
/linkedinSoham Jadhav| /codeLeetCode| /githubGitHub| /envel⌢pesohamjadhavth@gmail.com| ♂¶obile+91 83690 73590
Education
Sardar Patel Institute of T echnology, Mumbai 2023 – Present
Bachelor of Technology CGP A: 8.03
Shubham Raje Junior College (SRJC), Thane 2023
Higher Secondary Certificate (HSC)85.33%
V asant Vihar High School (VVHS), Thane 2021
Secondary School Certificate (SSC)89.80%
Projects
CityPulse — Civic Issue Management Platform — Next.js, React, TypeScript, MongoDB, T ailwind
CSS, NextAuth.js, Cloudinary , V ercel
•Built a civic issue management system using Next.js and MongoDB to aggregate duplicate complaints by
location and department, leading to consolidated and demand-driven issue tracking.
•Implemented role-based backend workflows using Next.js serverless APIs and database-backed access control
for citizens, authorities, and staff, leading to structured ownership and traceable issue resolution.
•Deployed the complete application on Vercel with envir

In [46]:
extracted_data

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-21T05:15:48+00:00', 'author': '', 'keywords': '', 'moddate': '2026-02-21T05:15:48+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\Soham_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Soham Rajesh Jadhav\n/linkedinSoham Jadhav| /codeLeetCode| /githubGitHub| /envel⌢pesohamjadhavth@gmail.com| ♂¶obile+91 83690 73590\nEducation\nSardar Patel Institute of T echnology, Mumbai 2023 – Present\nBachelor of Technology CGP A: 8.03\nShubham Raje Junior College (SRJC), Thane 2023\nHigher Secondary Certificate (HSC)85.33%\nV asant Vihar High School (VVHS), Thane 2021\nSecondary School Certificate (SSC)89.80%\nProjects\nCityPulse — Civic Issue Management Platform — Next.js, React, TypeScript, MongoDB, T ailwind\nCSS, NextAuth.js,

In [47]:
len(extracted_data)

1

In [48]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [49]:
minimal_docs=filter_to_minimal_docs(extracted_data)

In [50]:
minimal_docs

[Document(metadata={'source': '..\\data\\Soham_resume.pdf'}, page_content='Soham Rajesh Jadhav\n/linkedinSoham Jadhav| /codeLeetCode| /githubGitHub| /envel⌢pesohamjadhavth@gmail.com| ♂¶obile+91 83690 73590\nEducation\nSardar Patel Institute of T echnology, Mumbai 2023 – Present\nBachelor of Technology CGP A: 8.03\nShubham Raje Junior College (SRJC), Thane 2023\nHigher Secondary Certificate (HSC)85.33%\nV asant Vihar High School (VVHS), Thane 2021\nSecondary School Certificate (SSC)89.80%\nProjects\nCityPulse — Civic Issue Management Platform — Next.js, React, TypeScript, MongoDB, T ailwind\nCSS, NextAuth.js, Cloudinary , V ercel\n•Built a civic issue management system using Next.js and MongoDB to aggregate duplicate complaints by\nlocation and department, leading to consolidated and demand-driven issue tracking.\n•Implemented role-based backend workflows using Next.js serverless APIs and database-backed access control\nfor citizens, authorities, and staff, leading to structured ownersh

In [51]:
#chunking 
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50,
    )
    texts_chunk=text_splitter.split_documents(minimal_docs)
    return texts_chunk 

In [52]:
texts_chunk=text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 11


In [53]:
from langchain_huggingface import HuggingFaceEmbeddings
def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        encode_kwargs={"normalize_embeddings": True}
    )
    return embeddings

embedding=download_embeddings()

In [54]:
embedding

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [55]:
vector=embedding.embed_query("Hello")
vector
print(len(vector))

384


In [56]:
from langchain_community.vectorstores import FAISS

In [57]:
vector_store = FAISS.from_documents(
    texts_chunk,
    embedding
)

In [58]:
vector_store.save_local("../faiss_index")

In [59]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.load_local(
    "../faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

In [60]:
print(vector_store.index.ntotal)

11


In [61]:
retriever = vector_store.as_retriever(
    search_type="similarity",   #similarity is working as cosinie as we have normalised embeddings
    search_kwargs={"k": 3}
)

In [62]:
query = "What are the achievements listed in the resume?"
docs = retriever.invoke(query)

print(docs[0].page_content[:500])

Shubham Raje Junior College (SRJC), Thane 2023
Higher Secondary Certificate (HSC)85.33%
V asant Vihar High School (VVHS), Thane 2021
Secondary School Certificate (SSC)89.80%
Projects
CityPulse — Civic Issue Management Platform — Next.js, React, TypeScript, MongoDB, T ailwind


In [63]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [64]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3
)

response = llm.invoke("Hello")
print(response.content)

Hello! How can I help you today?


In [65]:
for i, d in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---\n")
    print(d.page_content)


--- Chunk 1 ---

Shubham Raje Junior College (SRJC), Thane 2023
Higher Secondary Certificate (HSC)85.33%
V asant Vihar High School (VVHS), Thane 2021
Secondary School Certificate (SSC)89.80%
Projects
CityPulse — Civic Issue Management Platform — Next.js, React, TypeScript, MongoDB, T ailwind

--- Chunk 2 ---

Soham Rajesh Jadhav
/linkedinSoham Jadhav| /codeLeetCode| /githubGitHub| /envel⌢pesohamjadhavth@gmail.com| ♂¶obile+91 83690 73590
Education
Sardar Patel Institute of T echnology, Mumbai 2023 – Present
Bachelor of Technology CGP A: 8.03
Shubham Raje Junior College (SRJC), Thane 2023

--- Chunk 3 ---

•Built a full-stack Learning Management System inspired by Udemy using React JS and Tailwind CSS.
•Implemented role-based authentication for students and instructors with secure RESTful APIs.
•Developed features for course creation, enrollment, and progress tracking using React Router.


In [66]:
context = "\n\n".join([d.page_content for d in docs])

In [67]:
prompt = f"""
You are a resume assistant.

Rules:
1. Answer ONLY using the provided context.
2. Do NOT add external information.
3. If answer is missing, say "Not found in provided context."

Context:
{context}

Question:
{query}
"""

In [69]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [70]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

template = """You are a resume assistant. 
Use the following pieces of retrieved context from a candidate's resume to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context:
{context}

Question:
{question}

Answer:"""

custom_rag_prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | custom_rag_prompt
    | llm
    | StrOutputParser()
)

In [73]:
query = "What is the current cgpa of candidate?"
response = rag_chain.invoke(query)
print(response)

The candidate's current CGPA is 8.03. This is from their Bachelor of Technology program at Sardar Patel Institute of Technology, Mumbai.
